In [1]:
#импортируем библиотеки requests для работы с API запросами, pandas и warnings для указания некритических ошибок
import requests
import pandas as pd
import warnings
import time
from datetime import datetime, timedelta
import logging
import sys

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('gibdd_okato.log', mode='a', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger()


In [3]:
# Установим geopy для геокодирования
!pip install geopy

# Теперь импортируем геокодер Nominatim и интсрумент для контроля качества запросов Ratelimiter 
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter


In [4]:
pip install supabase

Note: you may need to restart the kernel to use updated packages.


In [5]:
from supabase import create_client, Client

In [6]:
#выведим функцию для игнорирования всех предупреждений во время выполнения команды 
warnings.filterwarnings('ignore')

In [7]:
#выведем параметры для запроса к API Википедии
params = {
    'action': 'parse',
    'page': 'Список_городов_России',
    'format': 'json',
    'prop': 'text',
    'contentmodel': 'wikitext'
}

In [8]:
#выведем строку, которая идентифицирует браузер, ОС и устройство, с которого делается запрос. {
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

In [9]:
#выведем GET запрос к API Википедии
response = requests.get(
    'https://ru.wikipedia.org/w/api.php',
    params=params, headers=headers
)

In [10]:
#проверим успешность запроса
if response.status_code == 200:
    print("Запрос успешен!")
    data = response.json()

Запрос успешен!


In [11]:
#извлечем HTML-контент из данных
html_content = data['parse']['text']['*']

In [12]:
#извлечем все таблицы из страницы в Википедии
tables = pd.read_html(html_content)

In [13]:
#сохраним первую таблицу
cities_table = tables[0]

In [14]:
#сохраним таблицу в CSV файл
cities_table.to_csv('cities_russia.csv', index=False, encoding='utf-8')
print(f"Сохранено {len(cities_table)} городов!")

Сохранено 1125 городов!


In [15]:
#сохраним таблцу в базу данных
df = pd.read_csv('cities_russia.csv')

Переменуем колонки в таблице для удобства:

In [16]:
old_cols = df.columns.tolist()

In [17]:
new_cols = ['num', 'sign', 'city_name', 'region', 'federal', 'population', 'origin', 'status', 'old_names']

In [18]:
change_dict = dict(zip(old_cols, new_cols))

In [19]:
df = df.rename(columns=change_dict)

In [20]:
df = df[['city_name', 'region', 'federal', 'population']]

In [21]:
df['city_name'] = df['city_name'].str.replace('не призн.', '')

In [22]:
df['population'] = df['population'].str.replace(' ', '').str.split('[').str[0].astype('int')

In [23]:
#Оставим в таблице города с населением больше 100 000 чел., то есть только крупные города
to_work = df[df.population > 100000].reset_index(drop=True)

In [24]:
#добавим колонки для координат, пока что пустые
to_work['lat'], to_work['lon'] = None, None

Добавим координаты

In [25]:
# Создаем геокодер
geolocator = Nominatim(user_agent="city_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
# Создаем функцию для получения координат
def get_coordinates(city, region=None):
    try:
        # Пробуем искать с регионом для большей точности
        if region:
            location = geocode(f"{city}, {region}, Россия")
        else:
            location = geocode(f"{city}, Россия")
        
        if location:
            return location.latitude, location.longitude
    except:
        pass
    return None, None

# Применяем функцию к DataFrame
to_work['lat'] = None
to_work['lon'] = None

for idx, row in to_work.iterrows():
    lat, lon = get_coordinates(row['city_name'], row['region'])
    to_work.at[idx, 'lat'] = lat
    to_work.at[idx, 'lon'] = lon
    # Пауза чтобы не превысить лимиты API
    time.sleep(0.1)
    
    if idx % 10 == 0:
        print(f"Обработано {idx+1} городов")

Обработано 1 городов
Обработано 11 городов
Обработано 21 городов
Обработано 31 городов
2026-04-09 12:14:02,348 - WARNING - Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='nominatim.openstreetmap.org', port=443): Read timed out. (read timeout=1)")': /search?q=%D0%94%D0%BE%D0%BC%D0%BE%D0%B4%D0%B5%D0%B4%D0%BE%D0%B2%D0%BE%2C+%D0%9C%D0%BE%D1%81%D0%BA%D0%BE%D0%B2%D1%81%D0%BA%D0%B0%D1%8F+%D0%BE%D0%B1%D0%BB%D0%B0%D1%81%D1%82%D1%8C%2C+%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D1%8F&format=json&limit=1
Обработано 41 городов
Обработано 51 городов
Обработано 61 городов
Обработано 71 городов
Обработано 81 городов
Обработано 91 городов
Обработано 101 городов
Обработано 111 городов
Обработано 121 городов
Обработано 131 городов
Обработано 141 городов
Обработано 151 городов
Обработано 161 городов
Обработано 171 городов


In [26]:
to_work

,city_name,region,federal,population,lat,lon
0,Абакан,Хакасия,Сибирский,184769,53.72068,91.440602
1,Альметьевск,Татарстан,Приволжский,163512,54.900501,52.296378
2,Ангарск,Иркутская область,Сибирский,221296,52.544317,103.888214
3,Арзамас,Нижегородская область,Приволжский,104908,55.385736,43.816523
4,Армавир,Краснодарский край,Южный,187177,44.999358,41.129406
...,...,...,...,...,...,...
167,Элиста,Калмыкия,Южный,102583,46.307329,44.269221
168,Энгельс,Саратовская область,Приволжский,225428,51.501377,46.123309
169,Южно-Сахалинск,Сахалинская область,Дальневосточный,181587,46.957427,142.727438
170,Якутск,Якутия,Дальневосточный,355443,62.027408,129.731979


Получим погодные данные для 2-х городов: Пензы и Иркутска

In [27]:
# Open-Meteo Archive API
def get_weather_archive(lat, lon, city):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    # ПРАВИЛЬНЫЕ ДАТЫ (прошлое, не будущее!)
    end_date = datetime.now() - timedelta(days=1)  # вчера
    start_date = end_date - timedelta(days=100)     # 90 дней назад (максимум)
    
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date.strftime('%Y-%m-%d'),
        'end_date': end_date.strftime('%Y-%m-%d'),
        'daily': ['temperature_2m_max', 'temperature_2m_min', 
                 'precipitation_sum', 'pressure_msl_mean',  # ИЗМЕНИЛОСЬ!
                 'wind_speed_10m_max', 'relative_humidity_2m_mean'],
        'timezone': 'auto'
    }
    
    print(f"Запрос для {city}: {params['start_date']} - {params['end_date']}")
    
    try:
        response = requests.get(url, params=params, timeout=30)
        print(f"Статус: {response.status_code}")
        
        if response.status_code == 200:
            data = response.json()
            
            if 'daily' in data:
                df = pd.DataFrame({
                    'date': data['daily']['time'],
                    'temp_max': data['daily']['temperature_2m_max'],
                    'temp_min': data['daily']['temperature_2m_min'],
                    'precipitation': data['daily']['precipitation_sum'],
                    'pressure': data['daily']['pressure_msl_mean'],  # ИЗМЕНИЛОСЬ!
                    'wind_speed': data['daily']['wind_speed_10m_max'],
                    'humidity': data['daily']['relative_humidity_2m_mean']
                })
                df['city'] = city
                return df
        else:
            print(f"Ошибка: {response.text[:200]}")
            
    except Exception as e:
        print(f"Ошибка запроса: {e}")
    
    return pd.DataFrame()

# Тестируем
cities = {
    'Пенза': (53.1953, 45.0183),    
    'Иркутск': (52.2864, 104.2806)
}

for city, coords in cities.items():
    print(f"\n{'='*50}")

    weather_df = get_weather_archive(coords[0], coords[1], city)
    print(f"{city}: {len(weather_df)} дней")
    if not weather_df.empty:
        print(weather_df.head())
        all_weather = pd.DataFrame()  # пустой DataFrame для всех данных

for city, coords in cities.items():
    weather_df = get_weather_archive(coords[0], coords[1], city)
    if not weather_df.empty:
        all_weather = pd.concat([all_weather, weather_df], ignore_index=True)

# Сохраняем один общий файл
all_weather.to_csv('weather_all_cities.csv', index=False, encoding='utf-8')
print(f"Всего сохранено: {len(all_weather)} строк")


Запрос для Пенза: 2025-12-29 - 2026-04-08
Статус: 200
Пенза: 101 дней
         date  temp_max  temp_min  precipitation  pressure  wind_speed  \
0  2025-12-29      -3.2      -5.8            1.0     999.0        17.2   
1  2025-12-30      -3.5      -5.2            2.5     994.0        15.3   
2  2025-12-31      -3.7      -8.6            1.8     994.0        23.0   
3  2026-01-01      -8.2     -12.6            0.7    1004.7        18.7   
4  2026-01-02      -8.9     -12.3            0.3    1018.5        22.8   

   humidity   city  
0        90  Пенза  
1        90  Пенза  
2        84  Пенза  
3        84  Пенза  
4        82  Пенза  

Запрос для Иркутск: 2025-12-29 - 2026-04-08
Статус: 200
Иркутск: 101 дней
         date  temp_max  temp_min  precipitation  pressure  wind_speed  \
0  2025-12-29     -17.2     -22.8            0.0    1035.4         3.8   
1  2025-12-30     -14.1     -26.7            0.0    1037.9        15.9   
2  2025-12-31     -16.1     -20.3            0.0    1032.3   

In [28]:
all_weather

,date,temp_max,temp_min,precipitation,pressure,wind_speed,humidity,city
0,2025-12-29,-3.2,-5.8,1.0,999.0,17.2,90,Пенза
1,2025-12-30,-3.5,-5.2,2.5,994.0,15.3,90,Пенза
2,2025-12-31,-3.7,-8.6,1.8,994.0,23.0,84,Пенза
3,2026-01-01,-8.2,-12.6,0.7,1004.7,18.7,84,Пенза
4,2026-01-02,-8.9,-12.3,0.3,1018.5,22.8,82,Пенза
...,...,...,...,...,...,...,...,...
197,2026-04-04,3.2,-2.0,0.5,1009.9,26.2,61,Иркутск
198,2026-04-05,3.5,-4.8,0.0,1019.2,12.7,45,Иркутск
199,2026-04-06,8.2,-2.8,0.1,1009.2,20.6,44,Иркутск
200,2026-04-07,8.9,-0.4,1.2,1008.7,20.7,67,Иркутск


In [29]:
all_weather.to_csv('weather_Penza_Irk.csv', index=False, encoding='utf-8')

In [30]:
# Предполагая, что to_work - это pandas DataFrame
# Если это не DataFrame, преобразуйте его:
if not isinstance(to_work, pd.DataFrame):
    to_work = pd.DataFrame(to_work)

# Просмотр структуры данных
print(f"Размер таблицы: {to_work.shape}")
print(f"Колонки: {to_work.columns.tolist()}")
print(to_work.head())

Размер таблицы: (172, 6)
Колонки: ['city_name', 'region', 'federal', 'population', 'lat', 'lon']
     city_name                 region      federal  population        lat  \
0       Абакан                Хакасия    Сибирский      184769   53.72068   
1  Альметьевск              Татарстан  Приволжский      163512  54.900501   
2      Ангарск      Иркутская область    Сибирский      221296  52.544317   
3      Арзамас  Нижегородская область  Приволжский      104908  55.385736   
4      Армавир     Краснодарский край        Южный      187177  44.999358   

          lon  
0   91.440602  
1   52.296378  
2  103.888214  
3   43.816523  
4   41.129406  


In [31]:

import json

def upload_via_rest(df, table_name="to_work"):
    """
    Загрузка данных через REST API
    """
    headers = {
        "apikey": SUPABASE_KEY,
        "Authorization": f"Bearer {SUPABASE_KEY}",
        "Content-Type": "application/json",
        "Prefer": "return=minimal"
    }
    
    # Конвертируем DataFrame в JSON
    records = df.where(pd.notnull(df), None).to_dict(orient='records')
    
    # URL для вставки
    url = f"{SUPABASE_URL}/rest/v1/{table_name}"
    
    # Загружаем по частям
    chunk_size = 1000
    for i in range(0, len(records), chunk_size):
        chunk = records[i:i + chunk_size]
        response = requests.post(url, headers=headers, json=chunk)
        
        if response.status_code in [200, 201]:
            print(f"Загружено {len(chunk)} записей")
        else:
            print(f"Ошибка: {response.status_code}, {response.text}")
            break
    
    return True

# upload_via_rest(to_work)

In [32]:
to_work.to_csv('cities.csv', index=False, encoding='utf-8')


In [33]:
weather_df.to_csv('Penza_Irkutsk.csv', index=False, encoding='utf-8')


In [34]:
def get_all_regions():
    now = datetime.now()
    year = now.year
    month = now.month - 1 if now.month > 1 else 12
    if month == 12:
        year -= 1
    
    logger.info("Получаем список регионов РФ...")  
    rf_payload = {
        "maptype": 1,
        "region": "877", 
        "date": f'["MONTHS:{month}.{year}"]',
        "pok": "1"
    }
    
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Content-Type': 'application/json'
    }
    
    try:
        response = requests.post(
            "http://stat.gibdd.ru/map/getMainMapData",
            json=rf_payload,
            headers=headers,
            timeout=30
        )
        
        if response.status_code != 200:
            logger.error(f"Ошибка: {response.status_code}")
            return
        
        result = response.json()
        metabase = json.loads(result["metabase"])
        maps_data = json.loads(metabase[0]["maps"])
        
        regions = []
        for region in maps_data:
            regions.append({
                "id": region["id"],
                "name": region["name"],
                "districts": []
            })
        
        logger.info(f"Найдено {len(regions)} регионов")
              
        for i, region in enumerate(regions, 1):
            print(f"[{i}/{len(regions)}] {region['name']} ({region['id']})...")
            
            region_payload = {
                "maptype": 1,
                "region": region["id"],
                "date": f'["MONTHS:{month}.{year}"]',
                "pok": "1"
            }
            
            try:
                reg_response = requests.post(
                    "http://stat.gibdd.ru/map/getMainMapData",
                    json=region_payload,
                    headers=headers,
                    timeout=30
                )
                
                if reg_response.status_code == 200:
                    reg_result = reg_response.json()
                    reg_metabase = json.loads(reg_result["metabase"])
                    reg_maps_data = json.loads(reg_metabase[0]["maps"])
                    
                    municipalities = []
                    for municipality in reg_maps_data:
                        municipalities.append({
                            "id": municipality["id"],
                            "name": municipality["name"]
                        })
                    
                    region["districts"] = municipalities
                    logger.info(f"найдено {len(municipalities)} муниципалитетов")
                else:
                    logger.error(f"ошибка {reg_response.status_code}")
                    
            except Exception as e:
                logger.error(f"ошибка: {e}")
            
            if i < len(regions):
                time.sleep(0.5)
        
        filename = "regions_all.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(regions, f, ensure_ascii=False, indent=2)
        
        total_municipalities = sum(len(r["districts"]) for r in regions)
        logger.info(f"Файл: {filename}")
        logger.info(f"Регионов: {len(regions)}")
        logger.info(f"Всего муниципалитетов: {total_municipalities}")
                
    except Exception as e:
        logger.error(f"Критическая ошибка: {e}")

if __name__ == "__main__":  
    get_all_regions()

2026-04-09 12:16:20,199 - INFO - Получаем список регионов РФ...
2026-04-09 12:16:20,350 - INFO - Найдено 90 регионов
[1/90] Ямало-Ненецкий АО (71140)...
2026-04-09 12:16:20,455 - INFO - найдено 13 муниципалитетов
[2/90] Ханты-Мансийский АО (71100)...
2026-04-09 12:16:21,040 - INFO - найдено 21 муниципалитетов
[3/90] Тюменская область (71)...
2026-04-09 12:16:21,632 - INFO - найдено 24 муниципалитетов
[4/90] Ненецкий АО (10011)...
2026-04-09 12:16:22,205 - INFO - найдено 1 муниципалитетов
[5/90] Еврейская автономная область (99)...
2026-04-09 12:16:22,793 - INFO - найдено 6 муниципалитетов
[6/90] Республика Саха (Якутия) (98)...
2026-04-09 12:16:23,398 - INFO - найдено 35 муниципалитетов
[7/90] Чувашская Республика (97)...
2026-04-09 12:16:23,985 - INFO - найдено 24 муниципалитетов
[8/90] Чеченская Республика (96)...
2026-04-09 12:16:24,576 - INFO - найдено 17 муниципалитетов
[9/90] Республика Хакасия (95)...
2026-04-09 12:16:25,164 - INFO - найдено 13 муниципалитетов
[10/90] Удмуртская

In [35]:


def get_dtp_cards(region_id, district_id, year, month, start=1, end=100):

    '''Получение полных карточек ДТП с пагинацией'''

    url = "http://stat.gibdd.ru/map/getDTPCardData"

    payload = {
        "data": {
            "date": [f"MONTHS:{month}.{year}"],
            "ParReg": region_id,
            "order": {"type": "1", "fieldName": "dat"},
            "reg": district_id,
            "ind": "1",
            "st": str(start),
            "en": str(end),
            "fil": {
                "isSummary": False  # Полные данные вместо сводных
            },
            "fieldNames": [
                "dat", "time", "coordinates", "infoDtp", "k_ul", "dor", "ndu",
                "k_ts", "ts_info", "pdop", "pog", "osv", "s_pch", "s_pog",
                "n_p", "n_pg", "obst", "sdor", "t_osv", "t_p", "t_s", "v_p", "v_v"
            ]
        }
    }

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept": "application/json",
        "Content-Type": "application/json"
    }

    try:
        # Двойное кодирование JSON требуется API ГИБДД
        request_data = {
            "data": json.dumps(payload["data"], separators=(',', ':'))
        }

        response = requests.post(
            url,
            json=request_data,
            headers=headers,
            timeout=30
        )

        if response.status_code == 200:
            response_data = json.loads(response.text)
            return json.loads(response_data["data"]).get("tab", [])
        else:
            print(f"Ошибка HTTP: {response.status_code}")
            return None

    except Exception as e:
        print(f"Ошибка при запросе данных: {str(e)}")
        return None

In [36]:
region_id_I = "25"           # Иркутская область (код региона)
region_name_I = "Иркутская область"
district_id_I = "25401"      # Иркутск
district_name_I = "Иркутск"
year = 2025
month = 12


dtp_data_I = get_dtp_cards(region_id_I, district_id_I, year, month)

In [37]:
accidents_I = []

for item in dtp_data_I:
    accidents_I.append({
        "kart_id": item["KartId"],
        "date": item["date"],
        "time": item["Time"],
        "district": item["District"],
        "dtp_type": item["DTP_V"],
        "pog": item["POG"],
        "ran": item["RAN"],
        "k_ts": item["K_TS"],
        "k_uch": item["K_UCH"],
        "emtp_number": item["emtp_number"]
    })
accidents_df_I = pd.DataFrame(accidents_I)

In [38]:
accidents_df_I

,kart_id,date,time,district,dtp_type,pog,ran,k_ts,k_uch,emtp_number
0,225349941,31.12.2025,11:55,Свердловский район,Наезд на пешехода,0,1,1,3,250033050
1,225349972,31.12.2025,16:30,Октябрьский район,Наезд на пешехода,0,1,1,2,250033121
2,225349906,30.12.2025,23:00,Свердловский район,Столкновение,0,2,2,2,250032980
3,225349894,30.12.2025,16:50,Куйбышевский район,Столкновение,0,1,2,2,250033097
4,225349898,30.12.2025,18:45,Свердловский район,Наезд на пешехода,0,2,2,3,250032970
...,...,...,...,...,...,...,...,...,...,...
66,225231770,03.12.2025,11:40,Октябрьский район,Столкновение,0,1,2,3,250030018
67,225257512,02.12.2025,21:00,Ленинский район,Наезд на пешехода,0,1,1,2,250031314
68,225222763,02.12.2025,23:48,Свердловский район,Наезд на пешехода,0,1,1,2,250029595
69,225386040,02.12.2025,15:30,Октябрьский район,Столкновение,0,1,2,3,250030054


In [39]:
# Создаю пустой список для создания датафрейма
vehicles_I = []

# Итерируемся по транспортным средствам (infoDtp - ts_info)
for item in dtp_data_I:
    for ts in item["infoDtp"]["ts_info"]:
        vehicles_I.append({
            "kart_id": item["KartId"],
            "ts_number": ts["n_ts"],
            "status": ts["ts_s"],
            "class": ts["t_ts"],
            "brand": ts["marka_ts"],
            "model": ts["m_ts"],
            "color": ts["color"],
            "drive_type": ts["r_rul"],
            "year": ts["g_v"],
            "defects": ts["t_n"],
            "ownership": ts["f_sob"],
            "owner_type": ts["o_pf"]
        })
vehicles_df_I = pd.DataFrame(vehicles_I)

In [40]:
vehicles_df_I

,kart_id,ts_number,status,class,brand,model,color,drive_type,year,defects,ownership,owner_type
0,225349941,1,Осталось на месте ДТП,Прочие легковые автомобили,HONDA,FIT,Серый,С передним приводом,2005,Технические неисправности отсутствуют,Частная собственность,Физические лица
1,225349972,1,Осталось на месте ДТП,"А-класс (особо малый) до 3,5 м",MAZDA,Mazda 6,Черный,С передним приводом,2012,Технические неисправности отсутствуют,Частная собственность,Физические лица
2,225349906,1,Осталось на месте ДТП,"А-класс (особо малый) до 3,5 м",TOYOTA,Corolla,Белый,С передним приводом,2002,Технические неисправности отсутствуют,Частная собственность,Физические лица
3,225349906,2,Осталось на месте ДТП,"А-класс (особо малый) до 3,5 м",JAC,Прочие модели Jac,Черный,С передним приводом,2023,Технические неисправности отсутствуют,Собственность субъектов Российской Федерации,Физические лица
4,225349894,1,Осталось на месте ДТП,Прочая спецтехника,Прочие марки легковых ТС,Прочие марки легковых ТС,Оранжевый,С задним приводом,2023,Технические неисправности отсутствуют,Частная собственность,"Юридические лица, являющиеся коммерческими орг..."
...,...,...,...,...,...,...,...,...,...,...,...,...
115,225222763,1,Осталось на месте ДТП,"В-класс (малый) до 3,9 м",TOYOTA,Carina,Белый,С передним приводом,1994,Иные неисправности,Частная собственность,Физические лица
116,225386040,1,Осталось на месте ДТП,Прочие легковые автомобили,TOYOTA,Прочие модели Toyota,Белый,С передним приводом,2007,Технические неисправности отсутствуют,Частная собственность,Физические лица
117,225386040,2,Осталось на месте ДТП,Прочие легковые автомобили,TOYOTA,Прочие модели Toyota,Иные цвета,С передним приводом,2004,Технические неисправности отсутствуют,Частная собственность,Физические лица
118,225220085,1,Осталось на месте ДТП,"А-класс (особо малый) до 3,5 м",HONDA,FIT,Белый,С передним приводом,2009,Технические неисправности отсутствуют,Частная собственность,Физические лица


In [41]:
# Создаю пустой список для создания датафрейма
participants_I = []

for item in dtp_data_I:
    # Добавляю карт ид для того, чтобы связать таблицу с остальными
    kart_id = item['KartId']
    # Участники из ts_uch (инфо дтп - тс инфо - третий уровень вложенности)
    for ts in item['infoDtp']['ts_info']:
        # смотрим есть ли информация об участниках (точнее добавляем только если есть информация)
        if 'ts_uch' in ts and ts['ts_uch']:
            # Итерируемся по участникам транспортных средств
            for uch in ts['ts_uch']:
                participants_I.append({
                    'kart_id': kart_id,
                    'n_ts': ts['n_ts'],
                    'n_uch': uch.get('N_UCH', ''),
                    'k_uch': uch['K_UCH'],
                    'pol': uch.get('POL', ''),
                    'v_st': uch.get('V_ST', ''),
                    'alco': uch.get('ALCO', ''),
                    's_t': uch.get('S_T', ''),
                    's_sm': uch.get('S_SM', ''),
                    'safety_belt': uch.get('SAFETY_BELT', ''),
                    's_seat_group': uch.get('S_SEAT_GROUP', ''),
                    'injured_card_id': uch.get('INJURED_CARD_ID', '')
                })

    # Участники из uchInfo (дтп инфо - второй уровень вложенности)
    for uch in item['infoDtp'].get('uchInfo', []):
        participants_I.append({
            'kart_id': kart_id,
            'n_ts': None,
            'n_uch': uch.get('N_UCH', ''),
            'k_uch': uch['K_UCH'],
            'pol': uch.get('POL', ''),
            'v_st': uch.get('V_ST', ''),
            'alco': uch.get('ALCO', ''),
            's_t': uch.get('S_T', ''),
            's_sm': uch.get('S_SM', ''),
            'safety_belt': None,
            's_seat_group': None,
            'injured_card_id': None
        })

participants_df_I = pd.DataFrame(participants_I)

In [42]:
participants_df_I

,kart_id,n_ts,n_uch,k_uch,pol,v_st,alco,s_t,s_sm,safety_belt,s_seat_group,injured_card_id
0,225349941,1,1,Водитель,Мужской,5,00,Не пострадал,Нет (не скрывался),Да,,
1,225349941,None,3,Пешеход,Женский,,00,Не пострадал,Нет (не скрывался),None,None,None
2,225349941,None,2,Пешеход,Женский,,00,Получил телесные повреждения с показанием к ле...,Нет (не скрывался),None,None,None
3,225349972,1,1,Водитель,Женский,8,00,Не пострадал,Нет (не скрывался),Да,,
4,225349972,None,2,Пешеход,Мужской,,00,Получил телесные повреждения с показанием к ле...,Нет (не скрывался),None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
161,225386040,2,3,Пассажир,Женский,,00,Получил телесные повреждения с показанием к ле...,Нет (не скрывался),Да,,
162,225386040,2,2,Водитель,Мужской,18,00,Не пострадал,Нет (не скрывался),Да,,
163,225220085,1,3,Пассажир,Женский,,00,Получил телесные повреждения с показанием к ле...,Нет (не скрывался),Да,,
164,225220085,1,1,Водитель,Мужской,3,00,Не пострадал,Нет (не скрывался),Да,,


In [43]:
locations_I = []

for item in dtp_data_I:
    info = item["infoDtp"]
    locations_I.append({
        "kart_id": item["KartId"],
        "ndu": ", ".join(info["ndu"]), # тут и строкой ниже в ответе списки - превращаем в строку
        "sdor": ", ".join(info["sdor"]),
        "city": info["n_p"],
        "street": info["street"],
        "house": info["house"],
        "km": info["km"],
        "m": info["m"],
        "road_type": info["k_ul"],
        "road_category": info["dor_z"],
        "coord_w": info["COORD_W"],
        "coord_l": info["COORD_L"],
        "objects_near": ", ".join(info["OBJ_DTP"])
    })
locations_df_I = pd.DataFrame(locations_I)

In [44]:
locations_df_I

,kart_id,ndu,sdor,city,street,house,km,m,road_type,road_category,coord_w,coord_l,objects_near
0,225349941,"Недостатки зимнего содержания, Отсутствие, пло...",Регулируемый пешеходный переход,г Иркутск,ул Лермонтова,82,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",52.267081,104.257604,"Нерегулируемый пешеходный переход, Многокварти..."
1,225349972,Недостатки зимнего содержания,Регулируемый пешеходный переход,г Иркутск,ул Советская,85,0,0,Магистральные улицы общегородского значения,"Местного значения (дорога местного значения, в...",52.279883,104.334416,"Остановка общественного транспорта, Школа либо..."
2,225349906,"Отсутствие, плохая различимость горизонтальной...",Нерегулируемый перекрёсток неравнозначных улиц...,г Иркутск,мкр Первомайский,14а,0,0,Магистральные улицы общегородского значения,"Местного значения (дорога местного значения, в...",52.260608,104.240711,Многоквартирные жилые дома
3,225349894,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),г Иркутск,ул Баррикад,2 10,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",52.295949,104.303834,Отсутствие в непосредственной близости объекто...
4,225349898,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),г Иркутск,мкр Университетский,80А,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",52.252231,104.248637,"Остановка общественного транспорта, Многокварт..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,225231770,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),г Иркутск,ул Байкальская,22А,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",52.278468,104.301705,"Нерегулируемый пешеходный переход, Нерегулируе..."
67,225257512,Недостатки зимнего содержания,Нерегулируемый пешеходный переход,г Иркутск,ул Трактовая,20т 1,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",52.324149,104.234663,Отсутствие в непосредственной близости объекто...
68,225222763,Не установлены,Нерегулируемый пешеходный переход,г Иркутск,ул Сергеева,3 8а,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",52.265124,104.232048,Крупный торговый объект (являющийся объектом м...
69,225386040,Не установлены,Регулируемый перекресток,г Иркутск,ул Партизанская,105,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",52.272996,104.308842,Многоквартирные жилые дома


In [45]:
region_id = "56"  # Пензенская область
region_name = "Пензенская область"
district_id = "56401"  # Пенза
district_name = "Пенза"
year = 2025
month = 12


dtp_data = get_dtp_cards(region_id, district_id, year, month)

In [46]:
accidents = []

for item in dtp_data:
    accidents.append({
        "kart_id": item["KartId"],
        "date": item["date"],
        "time": item["Time"],
        "district": item["District"],
        "dtp_type": item["DTP_V"],
        "pog": item["POG"],
        "ran": item["RAN"],
        "k_ts": item["K_TS"],
        "k_uch": item["K_UCH"],
        "emtp_number": item["emtp_number"]
    })
accidents_df = pd.DataFrame(accidents)

In [47]:
accidents_df

,kart_id,date,time,district,dtp_type,pog,ran,k_ts,k_uch,emtp_number
0,225349589,31.12.2025,11:15,Железнодорожный р-н,Столкновение,1,2,2,4,560015215
1,225360116,31.12.2025,11:45,Железнодорожный р-н,Столкновение,0,1,2,2,560015343
2,225349568,30.12.2025,17:30,Октябрьский р-н,Столкновение,0,5,2,6,560015191
3,225349292,30.12.2025,11:15,Ленинский,Столкновение,0,2,2,3,560015135
4,225286178,29.12.2025,23:00,Октябрьский р-н,Наезд на стоящее ТС,0,1,2,3,560015039
...,...,...,...,...,...,...,...,...,...,...
60,225230682,04.12.2025,17:15,Ленинский,Наезд на пешехода,0,1,1,2,560013764
61,225230702,04.12.2025,21:30,Железнодорожный р-н,Наезд на препятствие,0,1,1,2,560013771
62,225230698,04.12.2025,18:10,Октябрьский р-н,Наезд на пешехода,0,1,1,2,560013783
63,225359262,01.12.2025,08:45,Железнодорожный р-н,Съезд с дороги,0,1,1,1,560015326


In [48]:
# Создаю пустой список для создания датафрейма
vehicles = []

# Итерируемся по транспортным средствам (infoDtp - ts_info)
for item in dtp_data:
    for ts in item["infoDtp"]["ts_info"]:
        vehicles.append({
            "kart_id": item["KartId"],
            "ts_number": ts["n_ts"],
            "status": ts["ts_s"],
            "class": ts["t_ts"],
            "brand": ts["marka_ts"],
            "model": ts["m_ts"],
            "color": ts["color"],
            "drive_type": ts["r_rul"],
            "year": ts["g_v"],
            "defects": ts["t_n"],
            "ownership": ts["f_sob"],
            "owner_type": ts["o_pf"]
        })
vehicles_df = pd.DataFrame(vehicles)

In [49]:
vehicles_df

,kart_id,ts_number,status,class,brand,model,color,drive_type,year,defects,ownership,owner_type
0,225349589,1,Осталось на месте ДТП,"С-класс (малый средний, компактный) до 4,3 м",CHEVROLET,Прочие модели Chevrolet,Черный,С передним приводом,2008,Иные неисправности,Частная собственность,Физические лица
1,225349589,2,Осталось на месте ДТП,Рефрижераторы,HOWO,Прочие модели Howo,Белый,С задним приводом,2023,Технические неисправности отсутствуют,Коллективная собственность (общественных объед...,"Юридические лица, являющиеся коммерческими орг..."
2,225360116,1,Осталось на месте ДТП,"С-класс (малый средний, компактный) до 4,3 м",ВАЗ,Vesta (Веста),Белый,С передним приводом,2018,Технические неисправности отсутствуют,Частная собственность,Физические лица
3,225360116,2,Осталось на месте ДТП,"S-класс (высший, представительский класс) боле...",BMW,X5,Белый,Полноприводные,2010,Технические неисправности отсутствуют,Коллективная собственность (общественных объед...,"Юридические лица, являющиеся некоммерческими о..."
4,225349568,1,Осталось на месте ДТП,"С-класс (малый средний, компактный) до 4,3 м",RENAULT,Logan,Серый,С передним приводом,2015,Технические неисправности отсутствуют,Частная собственность,Физические лица
...,...,...,...,...,...,...,...,...,...,...,...,...
117,225230682,1,Осталось на месте ДТП,"В-класс (малый) до 3,9 м",ВАЗ,Granta (Гранта),Серый,С передним приводом,2018,Технические неисправности отсутствуют,Частная собственность,Физические лица
118,225230702,1,Осталось на месте ДТП,"С-класс (малый средний, компактный) до 4,3 м",NISSAN,Almera,Синий,С передним приводом,2017,Технические неисправности отсутствуют,Частная собственность,Физические лица
119,225230698,1,Осталось на месте ДТП,"С-класс (малый средний, компактный) до 4,3 м",Datsun,mi-DO,Черный,С передним приводом,2015,Технические неисправности отсутствуют,Частная собственность,Физические лица
120,225359262,1,Осталось на месте ДТП,"С-класс (малый средний, компактный) до 4,3 м",RENAULT,Logan,Серый,С передним приводом,2017,Технические неисправности отсутствуют,Частная собственность,Физические лица


In [50]:
# Создаю пустой список для создания датафрейма
participants = []

for item in dtp_data:
    # Добавляю карт ид для того, чтобы связать таблицу с остальными
    kart_id = item['KartId']
    # Участники из ts_uch (инфо дтп - тс инфо - третий уровень вложенности)
    for ts in item['infoDtp']['ts_info']:
        # смотрим есть ли информация об участниках (точнее добавляем только если есть информация)
        if 'ts_uch' in ts and ts['ts_uch']:
            # Итерируемся по участникам транспортных средств
            for uch in ts['ts_uch']:
                participants.append({
                    'kart_id': kart_id,
                    'n_ts': ts['n_ts'],
                    'n_uch': uch.get('N_UCH', ''),
                    'k_uch': uch['K_UCH'],
                    'pol': uch.get('POL', ''),
                    'v_st': uch.get('V_ST', ''),
                    'alco': uch.get('ALCO', ''),
                    's_t': uch.get('S_T', ''),
                    's_sm': uch.get('S_SM', ''),
                    'safety_belt': uch.get('SAFETY_BELT', ''),
                    's_seat_group': uch.get('S_SEAT_GROUP', ''),
                    'injured_card_id': uch.get('INJURED_CARD_ID', '')
                })

    # Участники из uchInfo (дтп инфо - второй уровень вложенности)
    for uch in item['infoDtp'].get('uchInfo', []):
        participants.append({
            'kart_id': kart_id,
            'n_ts': None,
            'n_uch': uch.get('N_UCH', ''),
            'k_uch': uch['K_UCH'],
            'pol': uch.get('POL', ''),
            'v_st': uch.get('V_ST', ''),
            'alco': uch.get('ALCO', ''),
            's_t': uch.get('S_T', ''),
            's_sm': uch.get('S_SM', ''),
            'safety_belt': None,
            's_seat_group': None,
            'injured_card_id': None
        })

participants_df = pd.DataFrame(participants)

In [51]:
participants_df

,kart_id,n_ts,n_uch,k_uch,pol,v_st,alco,s_t,s_sm,safety_belt,s_seat_group,injured_card_id
0,225349589,1,4,Пассажир,Мужской,,00,"Раненый, находящийся (находившийся) на стацион...",Нет (не скрывался),Да,,
1,225349589,1,3,Пассажир,Женский,,00,Скончался на месте ДТП до приезда скорой медиц...,Нет (не скрывался),Да,,
2,225349589,1,1,Водитель,Мужской,0,00,"Раненый, находящийся (находившийся) на стацион...",Нет (не скрывался),Да,,
3,225349589,2,2,Водитель,Мужской,13,00,Не пострадал,Нет (не скрывался),Да,,
4,225360116,1,1,Водитель,Мужской,28,00,Не пострадал,Нет (не скрывался),Да,,
...,...,...,...,...,...,...,...,...,...,...,...,...
174,225230698,1,2,Водитель,Женский,7,00,Не пострадал,Нет (не скрывался),Да,,
175,225230698,None,1,Пешеход,Женский,,00,"Раненый, находящийся (находившийся) на амбулат...",Нет (не скрывался),None,None,None
176,225359262,1,1,Водитель,Мужской,6,00,Получил телесные повреждения с показанием к ле...,Нет (не скрывался),Да,,
177,225220698,1,2,Водитель,Мужской,12,00,Не пострадал,Нет (не скрывался),Да,,


In [52]:
locations = []

for item in dtp_data:
    info = item["infoDtp"]
    locations.append({
        "kart_id": item["KartId"],
        "ndu": ", ".join(info["ndu"]), # тут и строкой ниже в ответе списки - превращаем в строку
        "sdor": ", ".join(info["sdor"]),
        "city": info["n_p"],
        "street": info["street"],
        "house": info["house"],
        "km": info["km"],
        "m": info["m"],
        "road_type": info["k_ul"],
        "road_category": info["dor_z"],
        "coord_w": info["COORD_W"],
        "coord_l": info["COORD_L"],
        "objects_near": ", ".join(info["OBJ_DTP"])
    })
locations_df = pd.DataFrame(locations)

In [53]:
locations_df

,kart_id,ndu,sdor,city,street,house,km,m,road_type,road_category,coord_w,coord_l,objects_near
0,225349589,Недостатки зимнего содержания,Перегон (нет объектов на месте ДТП),г Пенза,ул Ушакова,18,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",53.216306,45.191188,Административные здания
1,225360116,Не установлены,Нерегулируемый перекрёсток неравнозначных улиц...,г Пенза,ул Антонова,3,0,0,Магистральные улицы районного значения,"Местного значения (дорога местного значения, в...",53.186834,45.068761,Крупный торговый объект (являющийся объектом м...
2,225349568,Не установлены,Перегон (нет объектов на месте ДТП),г Пенза,пр-кт Победы,126а 1,0,0,Магистральные улицы общегородского значения,"Местного значения (дорога местного значения, в...",53.228839,44.933567,"Остановка общественного транспорта, Регулируем..."
3,225349292,Не установлены,Перегон (нет объектов на месте ДТП),г Пенза,ул Мира,86Е,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",53.191366,44.962921,Отсутствие в непосредственной близости объекто...
4,225286178,Отсутствие временных ТСОД в местах проведения ...,"Мост, эстакада, путепровод",г Пенза,ул Ряжская,8,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",53.217174,44.959949,Жилые дома индивидуальной застройки
...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,225230682,Не установлены,Нерегулируемый пешеходный переход,г Пенза,ул Советская,7,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",53.183735,45.010729,"Административные здания, Зоны отдыха, Автостоя..."
61,225230702,Не установлены,Перегон (нет объектов на месте ДТП),г Пенза,ул Измайлова,150,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",53.174751,45.097467,"Спортивные и развлекательные объекты, Объект с..."
62,225230698,"Отсутствие, плохая различимость горизонтальной...",Нерегулируемый пешеходный переход,г Пенза,,,0,0,Иные места,"Местного значения (дорога местного значения, в...",53.233610,44.877648,Крупный торговый объект (являющийся объектом м...
63,225359262,Не установлены,Перегон (нет объектов на месте ДТП),г Пенза,ул Измайлова,150 + 600 м,0,0,Улицы и дороги местного значения в жилой застр...,"Местного значения (дорога местного значения, в...",53.174315,45.083835,Спортивные и развлекательные объекты


In [54]:
accidents_df_I.to_csv('accidents_Irk.csv', index=False, encoding='utf-8')

In [55]:
vehicles_df_I.to_csv('vehicles_Irk.csv', index=False, encoding='utf-8')

In [56]:
participants_df_I.to_csv('participants_Irk.csv', index=False, encoding='utf-8')

In [57]:
locations_df_I.to_csv('locations_Irk.csv', index=False, encoding='utf-8')

In [58]:
accidents_df.to_csv('accidents_Pen.csv', index=False, encoding='utf-8')

In [59]:
vehicles_df.to_csv('vehicles_Pen.csv', index=False, encoding='utf-8')

In [60]:
participants_df.to_csv('participants_Pen.csv', index=False, encoding='utf-8')

In [61]:
locations_df.to_csv('locations_Pen.csv', index=False, encoding='utf-8')

In [62]:
%%writefile my_script.py

# импортируем
import pandas as pd
import random


# определяем переменные для всего скрипта
CITY_LIST = ['Moscow', 'SPB', 'Irkutsk', 'EKB', 'Karaganda']

# определяем функции
def weather_function(nums:int) -> list:
    return [random.randint(-40, 40) for _ in range(nums)]

def city_function(nums) -> list:
    return [random.choice(CITY_LIST) for _ in range(nums)]

def merge_city_weather(cities:list, weathers:list) -> pd.DataFrame:
    weather_city = list(zip(cities, weathers))
    df = pd.DataFrame(weather_city)
    return df


# эта часть управляет скриптом - то, что мы делаем при запуске скрипта
if __name__ == '__main__':
    weather = weather_function(3)
    print(weather)
    cities = city_function(3)
    print(cities)
    final_df = merge_city_weather(cities, weather)
    final_df.columns=['city', 'weather']
    print(final_df)

Overwriting my_script.py


In [63]:
!python my_script.py

[-39, 12, -32]
['Karaganda', 'SPB', 'Moscow']
        city  weather
0  Karaganda      -39
1        SPB       12
2     Moscow      -32
